<a href="https://colab.research.google.com/github/akoratpa3/LLM-Final-Project/blob/main/LLM_Final_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
os.chdir("/content")
if not os.path.exists("Interplay-LM-Reasoning"):
    !git clone https://github.com/Interplay-LM-Reasoning/Interplay-LM-Reasoning.git
%cd /content/Interplay-LM-Reasoning

%pip install -q transformers==4.55.0
!python -m pip install -q transformers==4.55.0

In [ ]:
   import transformers; print("notebook:", transformers.__version__)
   !python -c "import transformers; print('shell:', transformers.__version__)"

In [ ]:
from datasets import load_dataset
from itertools import islice

ds = load_dataset("Interplay-LM-Reasoning/composition", split="train", streaming=True)
examples = list(islice(ds, 5))
for k, v in examples[0].items():
    print(f"{k}: {v}\n")

In [ ]:
from huggingface_hub import list_repo_files

for repo in ["Interplay-LM-Reasoning/extrapolation_rl", "Interplay-LM-Reasoning/extrapolation_midtrain"]:
    files = list_repo_files(repo)
    folders = sorted({f.rsplit("/", 1)[0] for f in files if "/" in f})
    print(f"\n=== {repo} ===")
    for f in folders: print(" ", f)

In [ ]:
from huggingface_hub import snapshot_download
local = snapshot_download(
    "Interplay-LM-Reasoning/extrapolation_rl",
    allow_patterns=["id2-10_0.2easy_0.3medium_0.5hard/base/*"],
    local_dir="/content/ckpt",
)
base = "/content/ckpt/id2-10_0.2easy_0.3medium_0.5hard/base"
!ls -la {base}
!df -h /content

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained(base)
model = AutoModelForCausalLM.from_pretrained(base, torch_dtype=torch.float32).cuda().eval()

In [ ]:
from datasets import get_dataset_split_names
print(get_dataset_split_names("Interplay-LM-Reasoning/composition"))

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
#TO MAKE BASELINE COMPARISON SCORES
import sys
sys.path.insert(0, "/content/Interplay-LM-Reasoning")
from utils.text_preprocess import compose_text, _split_solution

def make_prompt(e):
    return compose_text(e).split("<solution>")[0] + "<solution>"


import re, json, os, torch
from itertools import islice
from datasets import load_dataset


N_PER_OP = 100
OPS = range(2, 21)
TEST_SET_PATH = "/content/drive/MyDrive/interplay/eval/test_set_final.jsonl"
os.makedirs(os.path.dirname(TEST_SET_PATH), exist_ok=True)

if not os.path.exists(TEST_SET_PATH):
    stream = load_dataset("Interplay-LM-Reasoning/composition", split="test", streaming=True)
    by_op = {op: [] for op in OPS}
    for ex in islice(stream, 500_000):
        if ex["op"] in by_op and len(by_op[ex["op"]]) < N_PER_OP:
            by_op[ex["op"]].append({k: ex[k] for k in ["id", "op", "problem", "question", "solution"]})
        if all(len(v) >= N_PER_OP for v in by_op.values()):
            break
    with open(TEST_SET_PATH, "w") as f:
        for op in OPS:
            for ex in by_op[op]:
                f.write(json.dumps(ex) + "\n")

test_set = [json.loads(l) for l in open(TEST_SET_PATH)]
print(f"Test set: {len(test_set)} problems")
print(repr(make_prompt(test_set[0])))


def generate_ar(model, tok, test_set, run_name, out_dir, batch_size=32, max_new_tokens=1536):
    device = next(model.parameters()).device
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"{run_name}.jsonl")
    with open(out_path, "w") as f:
        for i in range(0, len(test_set), batch_size):
            batch = test_set[i:i+batch_size]
            prompts = [make_prompt(e) for e in batch]
            enc = tok(prompts, return_tensors="pt", padding=True, return_token_type_ids=False).to(device)
            with torch.no_grad():
                 out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True,
                        pad_token_id=tok.pad_token_id,
                        stop_strings=["</answer>"], tokenizer=tok)
            texts = tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=False)
            for e, t in zip(batch, texts):
                f.write(json.dumps({"id": e["id"], "op": e["op"], "model": run_name,
                                    "sample_idx": 0, "gen_solution_answer": "<solution>" + t}) + "\n")
            print(f"{min(i+batch_size, len(test_set))}/{len(test_set)}", flush=True)
    return out_path


def gold_answer(sol):
    return _split_solution(sol)[1] or None
def pred_answer(text):
    m = re.search(r"<answer>\s*(-?[\d\.]+)", text)
    if m: return m.group(1).rstrip(".")
    nums = re.findall(r"-?\d+(?:\.\d+)?", text); return nums[-1] if nums else None



In [ ]:
gen_path = generate_ar(model, tok, test_set, run_name="ar-100m_base",
                       out_dir="/content/drive/MyDrive/interplay/results/generations")
print("\nSaved to", gen_path)

# gold = {e["id"]: gold_answer(e["solution"]) for e in test_set}
gens = [json.loads(l) for l in open(gen_path)]
rows = [(e["op"], pred_answer(g["gen_solution_answer"]) == gold_answer(e["solution"]))
        for e, g in zip(test_set, gens)]
bands = {"ID (2-10)": range(2, 11), "edge (11-14)": range(11, 15), "hard (15-20)": range(15, 21)}
for b, ops in bands.items():
    r = [ok for op, ok in rows if op in ops]
    print(f"{b}: {sum(r)}/{len(r)} = {sum(r)/len(r):.0%}")

In [ ]:
import gc, json, torch
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM

PRETRAIN = "id2-10_0.2easy_0.3medium_0.5hard"
RL_CHECKPOINTS = {
    "ar-100m_rl-op7-10":  f"{PRETRAIN}/rl/op7-10_uniform",
    "ar-100m_rl-op9-12":  f"{PRETRAIN}/rl/op9-12_uniform",
    "ar-100m_rl-op11-14": f"{PRETRAIN}/rl/op11-14_uniform",
    "ar-100m_rl-op17-20": f"{PRETRAIN}/rl/op17-20_uniform",
}
GEN_DIR = "/content/drive/MyDrive/interplay/results/generations"
bands = {"ID (2-10)": range(2, 11), "edge (11-14)": range(11, 15), "hard (15-20)": range(15, 21)}

for run_name, sub in RL_CHECKPOINTS.items():
    print(f"\n===== {run_name} =====")
    snapshot_download("Interplay-LM-Reasoning/extrapolation_rl",
                      allow_patterns=[f"{sub}/*"], local_dir="/content/ckpt")
    rl_model = AutoModelForCausalLM.from_pretrained(f"/content/ckpt/{sub}",
                                                    torch_dtype=torch.float32).cuda().eval()
    p = generate_ar(rl_model, tok, test_set, run_name=run_name, out_dir=GEN_DIR)

    gens = [json.loads(l) for l in open(p)]
    rows = [(e["op"], pred_answer(g["gen_solution_answer"]) == gold_answer(e["solution"]))
            for e, g in zip(test_set, gens)]
    for b, ops in bands.items():
        r = [ok for op, ok in rows if op in ops]
        print(f"{b}: {sum(r)}/{len(r)} = {sum(r)/len(r):.0%}")

    del rl_model; gc.collect(); torch.cuda.empty_cache()   # free GPU before the next checkpoint